In [1]:
!pip install volumentations-3D
!pip install segmentation_models_pytorch
!pip install batchgenerators

  Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [2]:
from __future__ import annotations
# ====================================================
# Directory settings
# ====================================================
import os
from pathlib import Path

OUTPUT_DIR = '/kaggle/working'

INPUT_DIR='/kaggle/input/blood-vessel-segmentation'

In [3]:
# ====================================================
# CFG
# ====================================================
class CFG:
    apex=True
    wandb = False
    competition = "HOA3D"
    backbone = 'resnet18d'
    max_grad_norm = 1000
    debug = False
    debug_train_size = 200
    scheduler = "cosine"
    num_warmup_steps=0
    epochs = 2
    decoder_lr = 0.005
    betas = (0.9, 0.999)
    batch_size = 4
    infer_batch_size = 4
    ckpt_name = 'unet3d-baseline'
    weight_decay = 0.1
    seed = 42
    print_freq=50
    eval_freq =1000
    eval_step_save_start_epoch=0
    train = True

if CFG.debug:
    CFG.epochs = 1

In [4]:
# ====================================================
# Library
# ====================================================
import os
import gc
import re
import ast
import sys
import copy
import json
import time
import math
import string
import pickle
import random
import joblib
import itertools
import glob
import warnings
from matplotlib import pyplot as plt
warnings.filterwarnings("ignore")

import scipy as sp
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.optim import Adam, SGD, AdamW
from torch.utils.data import DataLoader, Dataset
import timm
import segmentation_models_pytorch as smp

from PIL import Image
import cv2

from transformers import get_linear_schedule_with_warmup, get_cosine_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(seed=42)

# Dataset

In [6]:
def extract_3d_voxels_for_patches(image_3d, patch_size=(128, 128, 128), stride=(64, 64, 64)):
    """
    Extracts patches in an image and returns a list of lowest voxels.
    """
    patches = []

    for start_x in range(0, image_3d.shape[0] - patch_size[0], stride[0]):
        for start_y in range(0, image_3d.shape[1] - patch_size[1], stride[1]):
            for start_z in range(0, image_3d.shape[2] - patch_size[2], stride[2]):
                # Lowest voxel needed to extract patch
                lowest_voxel = (start_x,start_y,start_z)
                patches.append(lowest_voxel)
                
    return patches

def extract_patch_from_voxel(image_3d, lowest_voxel, patch_size=(128, 128, 128)):
    """
    Extracts a 3d patch given the lowest voxel.
    """
    end_x = lowest_voxel[0] + patch_size[0] 
    end_y = lowest_voxel[1] + patch_size[1]
    end_z = lowest_voxel[2] + patch_size[2]

    patch = image_3d[lowest_voxel[0]:end_x,
                     lowest_voxel[1]:end_y,
                     lowest_voxel[2]:end_z]

    return patch



def filter_empty_patches_by_voxel(lowest_voxels, mask, patch_size, threshold = 0):
    """
    Removes the lower voxels that produce empty label patches.
    """
    positive_voxels = []

    for lowest_voxel in tqdm(lowest_voxels):
        # Extract the patch
        patch = mask[lowest_voxel[0]:lowest_voxel[0]  + patch_size[0],
                         lowest_voxel[1]:lowest_voxel[1]  + patch_size[1],
                         lowest_voxel[2]:lowest_voxel[2]  + patch_size[2]]

        if np.count_nonzero(patch) > threshold:
            positive_voxels.append(lowest_voxel)


    return positive_voxels


def norm_by_percentile(volume, low=10, high=99.8, alpha=0.01):
    xmin = np.percentile(volume,low)
    xmax = np.percentile(volume,high)
    x = (volume-xmin)/(xmax-xmin)
    
    if 1:
        x[x>1]=(x[x>1]-1)*alpha +1
        x[x<0]=(x[x<0])*alpha
    #x = np.clip(x,0,1)
    return x


def norm_by_parameters(volume,xmin=73,xmax=85, alpha=0.01):
    x = (volume-xmin)/(xmax-xmin)
    if 1:
        x[x>1]=(x[x>1]-1)*alpha +1
        x[x<0]=(x[x<0])*alpha
    #x = np.clip(x,0,1)
    return x

In [27]:
class ValidKidney3DDataset(Dataset):
    def __init__(self, patches, masks,patch_size,stride_size,norm_params, transformations=None):
        self.patches = patches
        self.masks = masks
        self.patch_size = patch_size
        self.lowest_voxels = extract_3d_voxels_for_patches(patches, patch_size=patch_size, stride=stride_size)
        self.lowest_voxels = filter_empty_patches_by_voxel(self.lowest_voxels, masks, patch_size,threshold=0)
        self.std = norm_params[0]
        self.mean = norm_params[1]
        self.transformations = transformations
        
    def __len__(self):
        return len(self.lowest_voxels)

    def __getitem__(self, idx):
        image = extract_patch_from_voxel(self.patches,self.lowest_voxels[idx],patch_size=self.patch_size)
        mask = extract_patch_from_voxel(self.masks,self.lowest_voxels[idx],patch_size=self.patch_size)
        
        image = np.expand_dims(image,0)
        mask = np.expand_dims(mask,0)
        
        data = {'image':image,'mask': mask}
        if self.transformations:
            data = self.transformations(**data)
            
        #data['image']  = (data['image']  - data['image'] .min()) / (data['image'] .max() - data['image'] .min() + 0.0001)
        data['image']=(data['image'] - data['image'].mean()) / (data['image'].std() + 0.0001)
        #data['image']=(data['image'] - self.mean) / (self.std + 0.0001)

        data['image'] = torch.tensor(data['image'], dtype=torch.float)
        data['mask'] = torch.tensor(data['mask'], dtype=torch.float)
        return data

In [28]:
class TrainKidney3DDataset(Dataset):
    def __init__(self, patches, masks, lowest_voxels, patch_size, norm_params, transformations=None):
        self.patches = patches
        self.masks = masks
        self.lowest_voxels = lowest_voxels
        self.patch_size = patch_size
        self.transformations = transformations
        self.std = norm_params[0]
        self.mean = norm_params[1]
        
    def __len__(self):
        return len(self.lowest_voxels)

    def __getitem__(self, idx):
        image = extract_patch_from_voxel(self.patches,self.lowest_voxels[idx],patch_size=self.patch_size)
        mask = extract_patch_from_voxel(self.masks,self.lowest_voxels[idx], patch_size=self.patch_size)
        
        image = np.expand_dims(image,0)
        mask = np.expand_dims(mask,0)
    
        data = {'image':image,'mask': mask}
        if self.transformations:
            data = self.transformations(**data)
        
        
        #data['image']  = (data['image']  - data['image'] .min()) / (data['image'] .max() - data['image'] .min() + 0.0001)
        data['image']=(data['image'] - data['image'].mean()) / (data['image'].std() + 0.0001)
        #data['image']=(data['image'] - self.mean) / (self.std + 0.0001)
        data['image'] = torch.tensor(data['image'], dtype=torch.float)
        data['mask'] = torch.tensor(data['mask'], dtype=torch.float)
        return data

In [9]:
import volumentations as volumen

def get_augmentation(patch_size):
    return volumen.Compose([
        volumen.RandomGamma(gamma_limit=(80, 120), p=0.3),
        volumen.GaussianNoise(var_limit=(0, 5), p=0.3),
        volumen.Flip(1, p=0.3),
        volumen.Flip(2, p=0.3),
    ], p=1.0)

In [10]:
kidney3_dense_paths =  sorted(glob.glob(f'/kaggle/input/blood-vessel-segmentation/train/kidney_3_sparse/images/*.tif'))
kidney3_dense_mask_paths =  sorted(glob.glob(f'/kaggle/input/blood-vessel-segmentation/train/kidney_3_dense/labels/*.tif'))
kidney3_dense_paths = [img for img in kidney3_dense_paths if any(os.path.splitext(os.path.basename(mask))[0] in img for mask in kidney3_dense_mask_paths)]

kidney1_dense_paths =  sorted(glob.glob(f'{INPUT_DIR}/train/kidney_1_dense/images/*.tif'))
kidney1_dense_mask_paths =  sorted(glob.glob(f'{INPUT_DIR}/train/kidney_1_dense/labels/*.tif'))

In [11]:
patch_size = (128,128,128)
stride = (32,32,32)

In [12]:
create_lowest_voxel_file = False
if create_lowest_voxel_file:
    volume = [cv2.imread(f, cv2.IMREAD_GRAYSCALE) for f in kidney1_dense_paths]
    volume = np.stack(volume).astype(np.uint8)

    mask = [cv2.imread(f, cv2.IMREAD_GRAYSCALE) for f in kidney1_dense_mask_paths]
    mask = np.stack(mask).astype(np.uint8)
    #mask = np.where(mask > 0.5, 1, 0).astype(np.uint8) # this is probably unnecessary
    my_voxels = extract_3d_voxels_for_patches(volume,patch_size=patch_size,stride=(32,32,32))
    my_voxels = filter_empty_patches_by_voxel(my_voxels,mask,patch_size=patch_size,threshold=50)
    np.save('/kaggle/working/kidney1_lowest_voxel_patches_stride32_thresh50.npy', my_voxels)

In [13]:
use_int16 = True

if use_int16:
    kidney3 = np.load('/kaggle/input/blood-vessel-segmentation-dense/kidney_3_dense.npz')
    kidney3_volume = kidney3['images']
    kidney3 = np.load('/kaggle/input/sennet-hoa-all-kidneys-dense/kidney_3_dense.npz')
    kidney3_masks = kidney3['mask']
    
    kidney1 = np.load('/kaggle/input/blood-vessel-segmentation-dense/kidney_1_dense.npz')
    kidney1_volume = kidney1['images']
    kidney1 = np.load('/kaggle/input/sennet-hoa-all-kidneys-dense/kidney_1_dense.npz')
    kidney1_masks = kidney1['mask']
else:
    kidney3 = np.load('/kaggle/input/sennet-hoa-all-kidneys-dense/kidney_3_dense.npz')
    kidney3_volume = kidney3['volume'].astype(np.uint8)
    kidney3_masks = kidney3['mask'].astype(np.uint8)

    kidney1 = np.load('/kaggle/input/sennet-hoa-all-kidneys-dense/kidney_1_dense.npz')
    kidney1_volume = kidney1['volume'].astype(np.uint8)
    kidney1_masks = kidney1['mask'].astype(np.uint8)

In [14]:
# precomputed for normalization
kidney1_std = 2747.315335558554
kidney1_mean = 23165.613198187508
kidney3_std = 758.3516133619871
kidney3_mean = 19570.52992715221

In [15]:
# patch size 128x128x128 stride 32x32x32, pos filter thres 50
loaded_lowest_voxels = np.load('/kaggle/input/sennet-hoa-kidney-patches/kidney1_lowest_voxel_patches_stride32_thresh50.npy') 
loaded_lowest_voxels.shape

(36872, 3)

In [16]:
train_dataset = TrainKidney3DDataset(
    patches=kidney1_volume,
    masks=kidney1_masks,
    lowest_voxels=loaded_lowest_voxels,
    patch_size=patch_size,
    transformations=get_augmentation(patch_size),
    norm_params = (kidney1_std,kidney1_mean)
)

In [17]:
len(train_dataset)

36872

# Model

In [18]:
n_blocks = 4
out_dim = 1
class TimmSegModel(nn.Module):
    def __init__(self, backbone, segtype='unet', pretrained=False):
        super(TimmSegModel, self).__init__()

        self.encoder = timm.create_model(
            backbone,
            in_chans=1,
            features_only=True,
            drop_rate=0,
            drop_path_rate=0,
            pretrained=pretrained
        )
        
        
        g = self.encoder(torch.rand(1, 1, 64, 64))
        encoder_channels = [1] + [_.shape[1] for _ in g]
        print(encoder_channels)
        decoder_channels = [256, 128, 64, 32, 16]
        if segtype == 'unet':
            self.decoder = smp.decoders.unet.decoder.UnetDecoder(
                encoder_channels=encoder_channels[:n_blocks+1],
                decoder_channels=decoder_channels[:n_blocks],
                n_blocks=n_blocks,
            )

        self.segmentation_head = nn.Conv2d(decoder_channels[n_blocks-1], out_dim, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

    def forward(self,x):
        global_features = [0] + self.encoder(x)[:n_blocks]
        seg_features = self.decoder(*global_features)
        seg_features = self.segmentation_head(seg_features)
        return seg_features

In [19]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, Optional, List


# Calculate symmetric padding for a convolution
def get_padding(kernel_size: int, stride: int = 1, dilation: int = 1, **_) -> int:
    padding = ((stride - 1) + dilation * (kernel_size - 1)) // 2
    return padding


# Calculate asymmetric TensorFlow-like 'SAME' padding for a convolution
def get_same_padding(x: int, k: int, s: int, d: int):
    return max((math.ceil(x / s) - 1) * s + (k - 1) * d + 1 - x, 0)


# Can SAME padding for given args be done statically?
def is_static_pad(kernel_size: int, stride: int = 1, dilation: int = 1, **_):
    return stride == 1 and (dilation * (kernel_size - 1)) % 2 == 0


# Dynamically pad input x with 'SAME' padding for conv with specified args
def pad_same(x, k: List[int], s: List[int], d: List[int] = (1, 1, 1), value: float = 0):
    ih, iw, iz = x.size()[-3:]
    pad_h = get_same_padding(ih, k[0], s[0], d[0])
    pad_w = get_same_padding(iw, k[1], s[1], d[1])
    pad_z = get_same_padding(iz, k[2], s[2], d[2])
    if pad_h > 0 or pad_w > 0 or pad_z > 0:
        x = F.pad(x, [pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_z // 2, pad_z - pad_z // 2], value=value)
    return x


def get_padding_value(padding, kernel_size, **kwargs) -> Tuple[Tuple, bool]:
    dynamic = False
    if isinstance(padding, str):
        # for any string padding, the padding will be calculated for you, one of three ways
        padding = padding.lower()
        if padding == 'same':
            # TF compatible 'SAME' padding, has a performance and GPU memory allocation impact
            if is_static_pad(kernel_size, **kwargs):
                # static case, no extra overhead
                padding = get_padding(kernel_size, **kwargs)
            else:
                # dynamic 'SAME' padding, has runtime/GPU memory overhead
                padding = 0
                dynamic = True
        elif padding == 'valid':
            # 'VALID' padding, same as padding=0
            padding = 0
        else:
            # Default to PyTorch style 'same'-ish symmetric padding
            padding = get_padding(kernel_size, **kwargs)
    return padding, dynamic


def conv3d_same(
        x, weight: torch.Tensor, bias: Optional[torch.Tensor] = None, stride: Tuple[int, int, int] = (1, 1, 1),
        padding: Tuple[int, int, int] = (0, 0, 0), dilation: Tuple[int, int, int] = (1, 1, 1), groups: int = 1):
    x = pad_same(x, weight.shape[-3:], stride, dilation)
    return F.conv3d(x, weight, bias, stride, (0, 0, 0), dilation, groups)


class Conv3dSame(nn.Conv3d):
    """ Tensorflow like 'SAME' convolution wrapper for 3d convolutions
    """

    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True):
        super(Conv3dSame, self).__init__(
            in_channels, out_channels, kernel_size, stride, 0, dilation, groups, bias)

    def forward(self, x):
        return conv3d_same(x, self.weight, self.bias, self.stride, self.padding, self.dilation, self.groups)


def create_conv3d_pad(in_chs, out_chs, kernel_size, **kwargs):
    padding = kwargs.pop('padding', '')
    kwargs.setdefault('bias', False)
    padding, is_dynamic = get_padding_value(padding, kernel_size, **kwargs)
    if is_dynamic:
        return Conv3dSame(in_chs, out_chs, kernel_size, **kwargs)
    else:
        return nn.Conv3d(in_chs, out_chs, kernel_size, padding=padding, **kwargs)

In [20]:
class MedNeXtBlock(nn.Module):

    def __init__(self, 
                in_channels:int, 
                out_channels:int, 
                exp_r:int=4, 
                kernel_size:int=7, 
                do_res:int=True,
                norm_type:str = 'group',
                n_groups:int or None = None,
                dim = '3d',
                grn = False
                ):

        super().__init__()

        self.do_res = do_res

        assert dim in ['2d', '3d']
        self.dim = dim
        if self.dim == '2d':
            conv = nn.Conv2d
        elif self.dim == '3d':
            conv = nn.Conv3d
            
        # First convolution layer with DepthWise Convolutions
        self.conv1 = conv(
            in_channels = in_channels,
            out_channels = in_channels,
            kernel_size = kernel_size,
            stride = 1,
            padding = kernel_size//2,
            groups = in_channels if n_groups is None else n_groups,
        )

        # Normalization Layer. GroupNorm is used by default.
        if norm_type=='group':
            self.norm = nn.GroupNorm(
                num_groups=in_channels, 
                num_channels=in_channels
                )
        elif norm_type=='layer':
            self.norm = LayerNorm(
                normalized_shape=in_channels, 
                data_format='channels_first'
                )

        # Second convolution (Expansion) layer with Conv3D 1x1x1
        self.conv2 = conv(
            in_channels = in_channels,
            out_channels = exp_r*in_channels,
            kernel_size = 1,
            stride = 1,
            padding = 0
        )
        
        # GeLU activations
        self.act = nn.GELU()
        
        # Third convolution (Compression) layer with Conv3D 1x1x1
        self.conv3 = conv(
            in_channels = exp_r*in_channels,
            out_channels = out_channels,
            kernel_size = 1,
            stride = 1,
            padding = 0
        )

        self.grn = grn
        if grn:
            if dim == '3d':
                self.grn_beta = nn.Parameter(torch.zeros(1,exp_r*in_channels,1,1,1), requires_grad=True)
                self.grn_gamma = nn.Parameter(torch.zeros(1,exp_r*in_channels,1,1,1), requires_grad=True)
            elif dim == '2d':
                self.grn_beta = nn.Parameter(torch.zeros(1,exp_r*in_channels,1,1), requires_grad=True)
                self.grn_gamma = nn.Parameter(torch.zeros(1,exp_r*in_channels,1,1), requires_grad=True)

 
    def forward(self, x, dummy_tensor=None):
        
        x1 = x
        x1 = self.conv1(x1)
        x1 = self.act(self.conv2(self.norm(x1)))
        if self.grn:
            # gamma, beta: learnable affine transform parameters
            # X: input of shape (N,C,H,W,D)
            if self.dim == '3d':
                gx = torch.norm(x1, p=2, dim=(-3, -2, -1), keepdim=True)
            elif self.dim == '2d':
                gx = torch.norm(x1, p=2, dim=(-2, -1), keepdim=True)
            nx = gx / (gx.mean(dim=1, keepdim=True)+1e-6)
            x1 = self.grn_gamma * (x1 * nx) + self.grn_beta + x1
        x1 = self.conv3(x1)
        if self.do_res:
            x1 = x + x1  
        return x1


class MedNeXtDownBlock(MedNeXtBlock):

    def __init__(self, in_channels, out_channels, exp_r=4, kernel_size=7, 
                do_res=False, norm_type = 'group', dim='3d', grn=False):

        super().__init__(in_channels, out_channels, exp_r, kernel_size, 
                        do_res = False, norm_type = norm_type, dim=dim,
                        grn=grn)

        if dim == '2d':
            conv = nn.Conv2d
        elif dim == '3d':
            conv = nn.Conv3d
        self.resample_do_res = do_res
        if do_res:
            self.res_conv = conv(
                in_channels = in_channels,
                out_channels = out_channels,
                kernel_size = 1,
                stride = 2
            )

        self.conv1 = conv(
            in_channels = in_channels,
            out_channels = in_channels,
            kernel_size = kernel_size,
            stride = 2,
            padding = kernel_size//2,
            groups = in_channels,
        )

    def forward(self, x, dummy_tensor=None):
        
        x1 = super().forward(x)
        
        if self.resample_do_res:
            res = self.res_conv(x)
            x1 = x1 + res

        return x1


class MedNeXtUpBlock(MedNeXtBlock):

    def __init__(self, in_channels, out_channels, exp_r=4, kernel_size=7, 
                do_res=False, norm_type = 'group', dim='3d', grn = False):
        super().__init__(in_channels, out_channels, exp_r, kernel_size,
                         do_res=False, norm_type = norm_type, dim=dim,
                         grn=grn)

        self.resample_do_res = do_res
        
        self.dim = dim
        if dim == '2d':
            conv = nn.ConvTranspose2d
        elif dim == '3d':
            conv = nn.ConvTranspose3d
        if do_res:            
            self.res_conv = conv(
                in_channels = in_channels,
                out_channels = out_channels,
                kernel_size = 1,
                stride = 2
                )

        self.conv1 = conv(
            in_channels = in_channels,
            out_channels = in_channels,
            kernel_size = kernel_size,
            stride = 2,
            padding = kernel_size//2,
            groups = in_channels,
        )


    def forward(self, x, dummy_tensor=None):
        
        x1 = super().forward(x)
        # Asymmetry but necessary to match shape
        
        if self.dim == '2d':
            x1 = torch.nn.functional.pad(x1, (1,0,1,0))
        elif self.dim == '3d':
            x1 = torch.nn.functional.pad(x1, (1,0,1,0,1,0))
        
        if self.resample_do_res:
            res = self.res_conv(x)
            if self.dim == '2d':
                res = torch.nn.functional.pad(res, (1,0,1,0))
            elif self.dim == '3d':
                res = torch.nn.functional.pad(res, (1,0,1,0,1,0))
            x1 = x1 + res

        return x1


class OutBlock(nn.Module):

    def __init__(self, in_channels, n_classes, dim):
        super().__init__()
        
        if dim == '2d':
            conv = nn.ConvTranspose2d
        elif dim == '3d':
            conv = nn.ConvTranspose3d
        self.conv_out = conv(in_channels, n_classes, kernel_size=1)
    
    def forward(self, x, dummy_tensor=None): 
        return self.conv_out(x)


class LayerNorm(nn.Module):
    """ LayerNorm that supports two data formats: channels_last (default) or channels_first. 
    The ordering of the dimensions in the inputs. channels_last corresponds to inputs with 
    shape (batch_size, height, width, channels) while channels_first corresponds to inputs 
    with shape (batch_size, channels, height, width).
    """
    def __init__(self, normalized_shape, eps=1e-5, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))        # beta
        self.bias = nn.Parameter(torch.zeros(normalized_shape))         # gamma
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError 
        self.normalized_shape = (normalized_shape, )
    
    def forward(self, x, dummy_tensor=False):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None, None] * x + self.bias[:, None, None, None]
            return x

In [ ]:
import torch.utils.checkpoint as checkpoint

class MedNeXt(nn.Module):

    def __init__(self, 
        in_channels: int, 
        n_channels: int,
        n_classes: int, 
        exp_r: int = 4,                            # Expansion ratio as in Swin Transformers
        kernel_size: int = 7,                      # Ofcourse can test kernel_size
        enc_kernel_size: int = None,
        dec_kernel_size: int = None,
        do_res: bool = False,                       # Can be used to individually test residual connection
        do_res_up_down: bool = False,             # Additional 'res' connection on up and down convs
        checkpoint_style: bool = None,            # Either inside block or outside block
        block_counts: list = [2,2,2,2,2,2,2,2,2], # Can be used to test staging ratio: 
                                            # [3,3,9,3] in Swin as opposed to [2,2,2,2,2] in nnUNet
        norm_type = 'group',
        dim = '3d',                                # 2d or 3d
        grn = False
    ):

        super().__init__()

        assert checkpoint_style in [None, 'outside_block']
        self.inside_block_checkpointing = False
        self.outside_block_checkpointing = False
        if checkpoint_style == 'outside_block':
            self.outside_block_checkpointing = True
        assert dim in ['2d', '3d']
        
        if kernel_size is not None:
            enc_kernel_size = kernel_size
            dec_kernel_size = kernel_size

        if dim == '2d':
            conv = nn.Conv2d
        elif dim == '3d':
            conv = nn.Conv3d
            
        self.stem = conv(in_channels, n_channels, kernel_size=1)
        if type(exp_r) == int:
            exp_r = [exp_r for i in range(len(block_counts))]
        
        self.enc_block_0 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels,
                out_channels=n_channels,
                exp_r=exp_r[0],
                kernel_size=enc_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                ) 
            for i in range(block_counts[0])]
        ) 

        self.down_0 = MedNeXtDownBlock(
            in_channels=n_channels,
            out_channels=2*n_channels,
            exp_r=exp_r[1],
            kernel_size=enc_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim
        )
    
        self.enc_block_1 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*2,
                out_channels=n_channels*2,
                exp_r=exp_r[1],
                kernel_size=enc_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[1])]
        )

        self.down_1 = MedNeXtDownBlock(
            in_channels=2*n_channels,
            out_channels=4*n_channels,
            exp_r=exp_r[2],
            kernel_size=enc_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )

        self.enc_block_2 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*4,
                out_channels=n_channels*4,
                exp_r=exp_r[2],
                kernel_size=enc_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[2])]
        )

        self.down_2 = MedNeXtDownBlock(
            in_channels=4*n_channels,
            out_channels=8*n_channels,
            exp_r=exp_r[3],
            kernel_size=enc_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )
        
        self.enc_block_3 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*8,
                out_channels=n_channels*8,
                exp_r=exp_r[3],
                kernel_size=enc_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )            
            for i in range(block_counts[3])]
        )
        
        self.down_3 = MedNeXtDownBlock(
            in_channels=8*n_channels,
            out_channels=16*n_channels,
            exp_r=exp_r[4],
            kernel_size=enc_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )

        self.bottleneck = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*16,
                out_channels=n_channels*16,
                exp_r=exp_r[4],
                kernel_size=dec_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[4])]
        )

        self.up_3 = MedNeXtUpBlock(
            in_channels=16*n_channels,
            out_channels=8*n_channels,
            exp_r=exp_r[5],
            kernel_size=dec_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )

        self.dec_block_3 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*8,
                out_channels=n_channels*8,
                exp_r=exp_r[5],
                kernel_size=dec_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[5])]
        )

        self.up_2 = MedNeXtUpBlock(
            in_channels=8*n_channels,
            out_channels=4*n_channels,
            exp_r=exp_r[6],
            kernel_size=dec_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )

        self.dec_block_2 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*4,
                out_channels=n_channels*4,
                exp_r=exp_r[6],
                kernel_size=dec_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[6])]
        )

        self.up_1 = MedNeXtUpBlock(
            in_channels=4*n_channels,
            out_channels=2*n_channels,
            exp_r=exp_r[7],
            kernel_size=dec_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )

        self.dec_block_1 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels*2,
                out_channels=n_channels*2,
                exp_r=exp_r[7],
                kernel_size=dec_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[7])]
        )

        self.up_0 = MedNeXtUpBlock(
            in_channels=2*n_channels,
            out_channels=n_channels,
            exp_r=exp_r[8],
            kernel_size=dec_kernel_size,
            do_res=do_res_up_down,
            norm_type=norm_type,
            dim=dim,
            grn=grn
        )

        self.dec_block_0 = nn.Sequential(*[
            MedNeXtBlock(
                in_channels=n_channels,
                out_channels=n_channels,
                exp_r=exp_r[8],
                kernel_size=dec_kernel_size,
                do_res=do_res,
                norm_type=norm_type,
                dim=dim,
                grn=grn
                )
            for i in range(block_counts[8])]
        )

        self.out_0 = OutBlock(in_channels=n_channels, n_classes=n_classes, dim=dim)

        # Used to fix PyTorch checkpointing bug
        self.dummy_tensor = nn.Parameter(torch.tensor([1.]), requires_grad=True)  

        self.block_counts = block_counts


    def iterative_checkpoint(self, sequential_block, x):
        """
        This simply forwards x through each block of the sequential_block while
        using gradient_checkpointing. This implementation is designed to bypass
        the following issue in PyTorch's gradient checkpointing:
        https://discuss.pytorch.org/t/checkpoint-with-no-grad-requiring-inputs-problem/19117/9
        """
        for l in sequential_block:
            x = checkpoint.checkpoint(l, x, self.dummy_tensor)
        return x


    def forward(self, x):
        
        x = self.stem(x)
        if self.outside_block_checkpointing:
            x_res_0 = self.iterative_checkpoint(self.enc_block_0, x)
            x = checkpoint.checkpoint(self.down_0, x_res_0, self.dummy_tensor)
            x_res_1 = self.iterative_checkpoint(self.enc_block_1, x)
            x = checkpoint.checkpoint(self.down_1, x_res_1, self.dummy_tensor)
            x_res_2 = self.iterative_checkpoint(self.enc_block_2, x)
            x = checkpoint.checkpoint(self.down_2, x_res_2, self.dummy_tensor)
            x_res_3 = self.iterative_checkpoint(self.enc_block_3, x)
            x = checkpoint.checkpoint(self.down_3, x_res_3, self.dummy_tensor)

            x = self.iterative_checkpoint(self.bottleneck, x)


            x_up_3 = checkpoint.checkpoint(self.up_3, x, self.dummy_tensor)
            dec_x = x_res_3 + x_up_3 
            x = self.iterative_checkpoint(self.dec_block_3, dec_x)

            del x_res_3, x_up_3

            x_up_2 = checkpoint.checkpoint(self.up_2, x, self.dummy_tensor)
            dec_x = x_res_2 + x_up_2 
            x = self.iterative_checkpoint(self.dec_block_2, dec_x)
  
            del x_res_2, x_up_2

            x_up_1 = checkpoint.checkpoint(self.up_1, x, self.dummy_tensor)
            dec_x = x_res_1 + x_up_1 
            x = self.iterative_checkpoint(self.dec_block_1, dec_x)

            del x_res_1, x_up_1

            x_up_0 = checkpoint.checkpoint(self.up_0, x, self.dummy_tensor)
            dec_x = x_res_0 + x_up_0 
            x = self.iterative_checkpoint(self.dec_block_0, dec_x)
            del x_res_0, x_up_0, dec_x

            x = checkpoint.checkpoint(self.out_0, x, self.dummy_tensor)

        else:
            x_res_0 = self.enc_block_0(x)
            x = self.down_0(x_res_0)
            x_res_1 = self.enc_block_1(x)
            x = self.down_1(x_res_1)
            x_res_2 = self.enc_block_2(x)
            x = self.down_2(x_res_2)
            x_res_3 = self.enc_block_3(x)
            x = self.down_3(x_res_3)

            x = self.bottleneck(x)
      

            x_up_3 = self.up_3(x)
            dec_x = x_res_3 + x_up_3 
            x = self.dec_block_3(dec_x)

            del x_res_3, x_up_3

            x_up_2 = self.up_2(x)
            dec_x = x_res_2 + x_up_2 
            x = self.dec_block_2(dec_x)

            del x_res_2, x_up_2

            x_up_1 = self.up_1(x)
            dec_x = x_res_1 + x_up_1 
            x = self.dec_block_1(dec_x)

            del x_res_1, x_up_1

            x_up_0 = self.up_0(x)
            dec_x = x_res_0 + x_up_0 
            x = self.dec_block_0(dec_x)
            del x_res_0, x_up_0, dec_x

            x = self.out_0(x)


        return x

In [21]:
from timm.layers.conv2d_same import Conv2dSame

def convert_3d(module):

    module_output = module
    if isinstance(module, torch.nn.BatchNorm2d):
        module_output = torch.nn.BatchNorm3d(
            module.num_features,
            module.eps,
            module.momentum,
            module.affine,
            module.track_running_stats,
        )
        if module.affine:
            with torch.no_grad():
                module_output.weight = module.weight
                module_output.bias = module.bias
        module_output.running_mean = module.running_mean
        module_output.running_var = module.running_var
        module_output.num_batches_tracked = module.num_batches_tracked
        if hasattr(module, "qconfig"):
            module_output.qconfig = module.qconfig
            
    elif isinstance(module, Conv2dSame):
        module_output = Conv3dSame(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=module.kernel_size[0],
            stride=module.stride[0],
            padding=module.padding[0],
            dilation=module.dilation[0],
            groups=module.groups,
            bias=module.bias is not None,
        )
        module_output.weight = torch.nn.Parameter(module.weight.unsqueeze(-1).repeat(1,1,1,1,module.kernel_size[0]))

    elif isinstance(module, torch.nn.Conv2d):
        module_output = torch.nn.Conv3d(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=module.kernel_size[0],
            stride=module.stride[0],
            padding=module.padding[0],
            dilation=module.dilation[0],
            groups=module.groups,
            bias=module.bias is not None,
            padding_mode=module.padding_mode
        )
        module_output.weight = torch.nn.Parameter(module.weight.unsqueeze(-1).repeat(1,1,1,1,module.kernel_size[0]))

    elif isinstance(module, torch.nn.MaxPool2d):
        module_output = torch.nn.MaxPool3d(
            kernel_size=module.kernel_size,
            stride=module.stride,
            padding=module.padding,
            dilation=module.dilation,
            ceil_mode=module.ceil_mode,
        )
    elif isinstance(module, torch.nn.AvgPool2d):
        module_output = torch.nn.AvgPool3d(
            kernel_size=module.kernel_size,
            stride=module.stride,
            padding=module.padding,
            ceil_mode=module.ceil_mode,
        )

    for name, child in module.named_children():
        module_output.add_module(
            name, convert_3d(child)
        )
    del module

    return module_output

# Loss

In [22]:
class DiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):
        
        inputs = F.sigmoid(inputs)       
        
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        intersection = (inputs * targets).sum()                            
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)  
        
        return 1 - dice
    

class BCELoss(nn.Module):
    def __init__(self, smooth=1.0, pos_weight=1, device='cpu'):
        super(BCELoss, self).__init__()
        self.smooth = smooth
        self.pos_weight = pos_weight
        self.device = device

    def forward(self, inputs, targets):
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        pos_weight = torch.tensor([self.pos_weight]).to(self.device)
        loss = F.binary_cross_entropy_with_logits(inputs, targets,pos_weight=pos_weight)
        return loss
    

class FocalLoss(nn.modules.loss._WeightedLoss):

    def __init__(self, gamma=0, size_average=None, ignore_index=-100,
                 reduce=None, balance_param=1.0):
        super(FocalLoss, self).__init__(size_average)
        self.gamma = gamma
        self.size_average = size_average
        self.ignore_index = ignore_index
        self.balance_param = balance_param

    def forward(self, input, target):
        logpt = - F.binary_cross_entropy_with_logits(input, target)
        pt = torch.exp(logpt)

        focal_loss = -((1 - pt) ** self.gamma) * logpt
        balanced_focal_loss = self.balance_param * focal_loss
        return balanced_focal_loss
    
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, pos_weight=1, device='cpu'):
        super(DiceBCELoss, self).__init__()
        self.pos_weight = pos_weight
        self.device = device
        
    def forward(self, inputs, targets, smooth=1):
        
        inputs = F.sigmoid(inputs)       
        
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        intersection = (inputs * targets).sum()                            
        dice_loss = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)  
        
        pos_weight = torch.tensor([self.pos_weight]).to(self.device)
        bce_loss = F.binary_cross_entropy(inputs, targets, pos_weight=pos_weight,reduction='mean')
        Dice_BCE = (bce_loss + dice_loss)/2
        
        return Dice_BCE

# Train

In [23]:
class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count
    
def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (remain %s)' % (asMinutes(s), asMinutes(rs))

def train_fn(
    train_loader: DataLoader,
    valid_loader: DataLoader,
    model: nn.Module,
    criterion: nn.Module,
    optimizer,
    epoch: int,
    scheduler,
    device,
    best_score,
) -> torch.Tensor:


    model.train()
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.apex)
    losses = AverageMeter()
    start = end = time.time()
    global_step = 0
    for step, (batch) in enumerate(train_loader):
        
        #if step < 2000:
            #if step % 50 == 0:
                #print(step)
            #continue
        
        inputs = batch['image'].to(device)
        labels = batch['mask'].to(device)

        batch_size = labels.size(0)
        
        with torch.cuda.amp.autocast(enabled=CFG.apex):
            y_preds = model(inputs)
            loss = criterion(y_preds, labels)
            
        losses.update(loss.item(), batch_size)
        scaler.scale(loss).backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), CFG.max_grad_norm
        )
        #print(loss)
        #print(torch.isnan(y_preds).any().item())
        #print(torch.isnan(labels).any().item())

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        global_step += 1
        scheduler.step()
 
        end = time.time()
        
        if step % CFG.print_freq == 0 or step == (len(train_loader) - 1):
            print(
                "Epoch: [{0}][{1}/{2}] "
                "Elapsed {remain:s} "
                "Loss: {loss.val:.4f}({loss.avg:.4f}) "
                "Grad: {grad_norm:.4f}  "
                "LR: {lr:.8f}  ".format(
                    epoch + 1,
                    step,
                    len(train_loader),
                    remain=timeSince(start, float(step + 1) / len(train_loader)),
                    loss=losses,
                    grad_norm=grad_norm,
                    lr=scheduler.get_lr()[0],
                )
            )
            
            
                
        if CFG.eval_step_save_start_epoch <= epoch and (
            (step + 1) % CFG.eval_freq == 0
        ):
            val_loss = valid_fn(valid_loader, model, criterion, device)
            score = val_loss
            if score < best_score:
                best_score = score
                save_ckpt(model)
                print(f"Saving New Best Score Model")
            
    torch.cuda.empty_cache()
    gc.collect()

    return losses.avg, best_score


@torch.inference_mode()
def valid_fn(
    valid_loader: DataLoader, model: nn.Module, criterion: nn.Module, device
) -> tuple[torch.Tensor, np.ndarray]:
    
    losses = AverageMeter()
    model.eval()
    start = end = time.time()
    for step, (batch) in enumerate(valid_loader):
        inputs = batch['image'].to(device)
        labels = batch['mask'].to(device)
        batch_size = labels.size(0)
        
        y_preds = model(inputs)
            
        loss = criterion(y_preds, labels)
    
            
        losses.update(loss.item(), batch_size)
        end = time.time()
        if step % CFG.print_freq == 0 or step == (len(valid_loader) - 1):
            print(
                "EVAL: [{0}/{1}] "
                "Elapsed {remain:s} "
                "Loss: {loss.val:.4f}({loss.avg:.4f}) ".format(
                    step,
                    len(valid_loader),
                    loss=losses,
                    remain=timeSince(start, float(step + 1) / len(valid_loader)),
                )
            )


    model.train()
    return losses.avg

def save_ckpt(
    model: torch.nn.Module,
) -> None:

    save_path = OUTPUT_DIR + f'/{CFG.ckpt_name}.pth'

    torch.save(
        {"model": model.state_dict()},
        save_path,
    )

In [32]:
resnet_backbone = True
if resnet_backbone:
    model = TimmSegModel(CFG.backbone)
    model = convert_3d(model)

    model.to(device)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {total_params}")
else:
    model = MedNeXt(
        in_channels = 1, 
        n_channels = 32,
        n_classes = 1, 
        exp_r=[2,3,4,4,4,4,4,3,2],       
        kernel_size=3,         
        do_res=True,                     
        do_res_up_down = True,
        block_counts = [3,4,4,4,4,4,4,4,3],
        checkpoint_style = 'outside_block'
    )

model.to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params}")

[1, 64, 64, 128, 256, 512]
Trainable parameters: 39722049
Trainable parameters: 39722049


In [36]:
aug = get_augmentation(patch_size)
train_dataset = TrainKidney3DDataset(
    patches=kidney1_volume,
    masks=kidney1_masks,
    lowest_voxels=loaded_lowest_voxels,
    patch_size=patch_size,
    norm_params = (kidney1_std,kidney1_mean),
    transformations=get_augmentation(patch_size),
)

valid_dataset = ValidKidney3DDataset(
    patches=kidney3_volume,
    masks=kidney3_masks,
    patch_size=patch_size,
    stride_size=(64, 64, 64),
    norm_params = (kidney3_std,kidney3_mean)
)


train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=CFG.infer_batch_size, shuffle=False)

# Loss function and optimizer
#criterion = BCELoss(pos_weight=120,device=device)
criterion = DiceLoss()
optimizer = AdamW(model.parameters(), lr=CFG.decoder_lr)
num_train_steps = int(len(train_dataset) / CFG.batch_size * CFG.epochs)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_train_steps//2)

  0%|          | 0/3300 [00:00<?, ?it/s]

In [ ]:
num_train_steps

In [ ]:
best_score = np.inf
for epoch in range(CFG.epochs):
    start_time = time.time()
    avg_loss, best_score = train_fn(
    train_loader,
    valid_loader,
    model,
    criterion,
    optimizer,
    epoch,
    scheduler,
    device,
    best_score,
    )
            
            
    avg_val_loss = valid_fn(valid_loader, model, criterion, device)
    elapsed = time.time() - start_time
    save_ckpt(model)

Epoch: [1][0/9218] Elapsed 0m 2s (remain 316m 28s) Loss: 0.9829(0.9829) Grad: 15865.0996  LR: 0.00500000  
Epoch: [1][50/9218] Elapsed 1m 48s (remain 324m 7s) Loss: 0.5438(0.7824) Grad: 243425.7812  LR: 0.00499961  
Epoch: [1][100/9218] Elapsed 3m 33s (remain 321m 34s) Loss: 0.5184(0.7145) Grad: 46428.5391  LR: 0.00499849  
Epoch: [1][150/9218] Elapsed 5m 19s (remain 320m 6s) Loss: 0.8237(0.6639) Grad: 39996.2188  LR: 0.00499665  
Epoch: [1][200/9218] Elapsed 7m 5s (remain 317m 49s) Loss: 0.3181(0.6169) Grad: 60020.8984  LR: 0.00499408  
Epoch: [1][250/9218] Elapsed 8m 51s (remain 316m 21s) Loss: 0.3028(0.5885) Grad: 17618.0195  LR: 0.00499079  
Epoch: [1][300/9218] Elapsed 10m 37s (remain 314m 43s) Loss: 0.2939(0.5751) Grad: 24401.2402  LR: 0.00498677  
Epoch: [1][350/9218] Elapsed 12m 23s (remain 312m 54s) Loss: 0.7929(0.5541) Grad: 11821.7236  LR: 0.00498203  
Epoch: [1][400/9218] Elapsed 14m 9s (remain 311m 23s) Loss: 0.2080(0.5321) Grad: 7579.3379  LR: 0.00497657  
Epoch: [1][450/

In [34]:
gc.collect()
torch.cuda.empty_cache()

# Visualize Patches

In [ ]:
try:
    import pyvirtualdisplay
except: 
    print('pip ...')
    !pip install -q piglet pyvirtualdisplay
    !pip install -q vtk
    !pip install -q trame ipywidgets
    !pip install -q trame-vuetify
    !pip install -q pyvista  
    !apt-get install -y xvfb

    
import vtk
from pyvirtualdisplay import Display
display = Display(visible=0, size=(600, 400))
display.start()
print('IMPORT OK')

In [ ]:
# notebook plotting: https://docs.pyvista.org/version/stable/user-guide/jupyter/index.html
import pyvista as pv
#pv.set_jupyter_backend('trame')

In [ ]:
test_dataset = Kidney3DDataset(
    patches=kidney3_volume,
    masks=kidney3_masks,
    patch_size=patch_size,
    stride_size=(64, 64, 64),
    mode="train",
)

In [ ]:
mask_gt = test_dataset[0]['mask'].squeeze()
mask_gt.sum()

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/unet3d-baseline.pth",map_location=torch.device(device))['model'])
model.eval()
print("")

In [ ]:
model.eval()
print("")

In [ ]:
with torch.no_grad():
    with torch.cuda.amp.autocast():
        batch =test_dataset[1600]
        inputs = torch.unsqueeze(batch['image'].to(device),0)
        labels = batch['mask'].to(device)
        print(labels.sum())
        masks = model(inputs).squeeze()
        mask = F.sigmoid(masks).cpu().detach()
        print(mask.sum())

In [ ]:
block_mask = np.ones_like(kidney1_volume,dtype=np.uint8)

In [ ]:
block_mask, pixel = extract_3d_block(kidney1_masks, block_mask, block_size=(128, 128, 128), stride=(64, 64, 64), max_failures=2)
mypatch = extract_patch_from_pixel(kidney1_masks, pixel,patch_size=(128, 128, 128))

In [ ]:
block_mask, pixel = extract_3d_block(kidney1_masks, block_mask, block_size=(128, 128, 128), stride=(64, 64, 64), max_failures=2)
mypatch = extract_patch_from_pixel(kidney1_masks, pixel,patch_size=(128, 128, 128))

pl = pv.Plotter()
point1 = np.stack(np.where(mypatch>0)).T
pd1 = pv.PolyData(point1)
mesh1 = pd1.glyph(geom=pv.Cube())
pl.add_mesh(mesh1, color='red')
pl.show()

In [ ]:
pl = pv.Plotter()
point1 = np.stack(np.where(labels.detach().cpu().squeeze() > 0)).T
pd1 = pv.PolyData(point1)
mesh1 = pd1.glyph(geom=pv.Cube())
pl.add_mesh(mesh1, color='red')
pl.show()

In [ ]:
pl = pv.Plotter()
point1 = np.stack(np.where(mask > 0.1)).T
pd1 = pv.PolyData(point1)
mesh1 = pd1.glyph(geom=pv.Cube())
pl.add_mesh(mesh1, color='red')
pl.show()